# Bulk RNA-seq survival analysis — HLA-DRA expression and overall survival in LUAD

**Goal:** Demonstrate that HLA-DRA expression is a transcriptional biomarker of
overall survival in LUAD, and that this association remains significant after
adjustment for clinical covariates and driver mutations.

**Dataset:** ORIEN Avatar LUAD cohort, accessed via cBioPortal. Bulk RNA-seq
expression data (mRNA z-scores, RNA-seq V2 RSEM), overall survival, clinical
metadata, and whole exome sequencing mutation calls.

**Data availability:** Data is available through cBioPortal. See
`data/cbioportal_query.md` for query details. Raw files are not committed to
this repository — update paths in `paths_config.yml` to match your local copy.

**Figures produced:**
- Figure 1a — Kaplan-Meier survival curve, HLA-DRA high vs low (median split)
- Figure 1b — Multivariate Cox proportional hazards forest plot
- Supplemental figure S1a — Kaplan-Meier survival curve, MHC II gene cluster
  (hierarchical clustering on full MHC II gene set)

**Input:**
- mRNA z-score expression matrices (16 files, joined on sample ID)
- Overall survival and clinical metadata
- WES mutation calls for driver genes

**Output:** Publication figures saved to `outputs/figures/figure1/`

In [ ]:
# standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path
import yaml

# survival analysis
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.plotting import add_at_risk_counts
from lifelines.statistics import logrank_test
from matplotlib.offsetbox import AnchoredText

# clustering
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# utilities
pd.options.mode.chained_assignment = None

# figure settings
sns.set(font_scale=1.5)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# paper-wide color palettes
# MHC II High = orange (#FF8811), MHC II Low = purple (#462255)
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

In [ ]:
# load paths configuration
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# output paths
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']
fig_out   = fig_dir / 'figure1'
table_out = table_dir / 'figure1'
fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

# data root
orien_root = Path(cfg['datasets']['orien']['root'])

### Expression data

mRNA z-score matrices are split across 16 files due to cBioPortal export limits.
All files share the same gene columns and are joined on sample ID.

In [ ]:
# load expression data
# cBioPortal may split large downloads into multiple part files
# this handles both single-file and multi-part downloads
expr_path  = orien_root / cfg['datasets']['orien']['expression']
expr_parts = sorted(expr_path.parent.glob('mysig_pt*_mRNA expression z-Scores (RNA Seq V2 RSEM).txt'))

if expr_path.exists():
    # single file download
    expr_df = pd.read_csv(expr_path, sep='\t', index_col='SAMPLE_ID', na_values=['NP'])
    expr_df = expr_df.drop(columns=['STUDY_ID'], errors='ignore')
    print(f'Loaded single expression file: {expr_df.shape[0]} samples × {expr_df.shape[1]} genes')

elif len(expr_parts) > 0:
    # multi-part download
    print(f'Found {len(expr_parts)} expression part files — joining')
    chunks = []
    for f in expr_parts:
        chunk = pd.read_csv(f, sep='\t', index_col='SAMPLE_ID', na_values=['NP'])
        chunk = chunk.drop(columns=['STUDY_ID'], errors='ignore')
        chunks.append(chunk)
    expr_df = chunks[0].join(chunks[1:], how='outer')
    print(f'Expression matrix: {expr_df.shape[0]} samples × {expr_df.shape[1]} genes')

else:
    raise FileNotFoundError(
        f'No expression data found at {expr_path}\n'
        f'See data/cbioportal_query.md for download instructions'
    )

# drop genes with >5% missing values
missing_threshold = int(0.05 * len(expr_df))
genes_to_drop     = expr_df.isna().sum()[expr_df.isna().sum() > missing_threshold].index
expr_df           = expr_df.drop(columns=genes_to_drop)
print(f'Genes dropped (>5% missing): {len(genes_to_drop)}')
print(f'Final expression matrix: {expr_df.shape[0]} samples × {expr_df.shape[1]} genes')

In [ ]:
# reload without dropping anything — just look at the raw data
expr_raw = pd.read_csv(
    orien_root / cfg['datasets']['orien']['expression'],
    sep='\t', index_col='SAMPLE_ID', na_values=['NP']
)
expr_raw = expr_raw.drop(columns=['STUDY_ID'], errors='ignore')

print(f'Shape: {expr_raw.shape}')
print(f'\nColumn names:')
print(expr_raw.columns.tolist())
print(f'\nFirst 5 rows:')
expr_raw.head()

In [ ]:
# drop samples with no expression data — use HLA-DRA as indicator
# samples without RNA-seq data will have NaN for all genes
expr_df = expr_raw.dropna(subset=['HLA-DRA'])

print(f'Samples with HLA-DRA expression: {len(expr_df):,}')
print(f'Samples dropped (no RNA-seq):    {len(expr_raw) - len(expr_df):,}')
print(f'\nMissing values per gene after filtering:')
print(expr_df.isna().sum().sort_values(ascending=False))

In [ ]:
# drop remaining genes with >5% missing after filtering to RNA-seq samples
missing_threshold = int(0.05 * len(expr_df))
genes_to_drop     = expr_df.isna().sum()[expr_df.isna().sum() > missing_threshold].index
expr_df           = expr_df.drop(columns=genes_to_drop)

print(f'Samples retained:              {len(expr_df):,}')
print(f'Genes dropped (>5% missing):   {len(genes_to_drop)}')
print(f'Final expression matrix:       {expr_df.shape[0]} samples × {expr_df.shape[1]} genes')

In [ ]:
# load clinical metadata files
survival_df = pd.read_csv(
    orien_root / cfg['datasets']['orien']['survival'],
    sep='\t', index_col='Sample Id', na_values=['NP']
)

age_tmb_df = pd.read_csv(
    orien_root / cfg['datasets']['orien']['age_tmb'],
    sep='\t', index_col='Sample Id', na_values=['NP']
)

sex_treatment_df = pd.read_csv(
    orien_root / cfg['datasets']['orien']['sex_treatment'],
    sep='\t', index_col='Sample Id', na_values=['NP']
)

cancer_type_stage_df = pd.read_csv(
    orien_root / cfg['datasets']['orien']['cancer_type_stage'],
    sep='\t', index_col='Sample Id', na_values=['NP']
)

tmb_msi_df = pd.read_csv(
    orien_root / cfg['datasets']['orien']['tmb_msi'],
    sep='\t', index_col='Sample Id', na_values=['NP']
)

# combine all clinical metadata
clinical_df = pd.concat([
    survival_df,
    age_tmb_df,
    sex_treatment_df,
    cancer_type_stage_df,
    tmb_msi_df,
], axis=1)

print(f'Clinical metadata: {clinical_df.shape[0]} samples × {clinical_df.shape[1]} columns')
print(f'\nColumns:')
print(clinical_df.columns.tolist())

In [ ]:
# load mutation files
mut_ros1_egfr = pd.read_csv(
    orien_root / cfg['datasets']['orien']['mutations']['ros1_egfr'],
    sep='\t', index_col='Sample Id', na_values=['NP']
)

mut_alk_braf = pd.read_csv(
    orien_root / cfg['datasets']['orien']['mutations']['alk_braf'],
    sep='\t', index_col='Sample Id', na_values=['NP']
)

mut_met_ret = pd.read_csv(
    orien_root / cfg['datasets']['orien']['mutations']['met_ret'],
    sep='\t', index_col='Sample Id', na_values=['NP']
)

mut_kras_tp53 = pd.read_csv(
    orien_root / cfg['datasets']['orien']['mutations']['kras_tp53'],
    sep='\t', index_col='Sample Id', na_values=['NP']
)

# combine all mutation data
mutations_df = pd.concat([
    mut_ros1_egfr,
    mut_alk_braf,
    mut_met_ret,
    mut_kras_tp53,
], axis=1)

print(f'Mutation data: {mutations_df.shape[0]} samples × {mutations_df.shape[1]} columns')
print(f'\nColumns:')
print(mutations_df.columns.tolist())

In [ ]:
# merge expression, clinical, and mutation data on sample ID
rna_meta_df = pd.concat([expr_df, clinical_df, mutations_df], axis=1)

print(f'Merged shape: {rna_meta_df.shape[0]} samples × {rna_meta_df.shape[1]} columns')
print(f'Samples with expression data: {rna_meta_df["HLA-DRA"].notna().sum():,}')
print(f'Samples with survival data:   {rna_meta_df["OverallSurvivalStatus"].notna().sum():,}')

In [ ]:
# subset to LUAD
luad_df = rna_meta_df[
    rna_meta_df['Cancer Type Details'] == '81403 Adenocarcinoma, NOS'
].copy()

print(f'LUAD samples: {len(luad_df):,}')
print(f'LUAD samples with expression: {luad_df["HLA-DRA"].notna().sum():,}')
print(f'LUAD samples with survival:   {luad_df["OverallSurvivalStatus"].notna().sum():,}')

In [ ]:
# stage grouping — early (I/II) vs late (III/IV)
def extract_staging_basic(val):
    val = str(val).strip().upper()
    match = re.match(r'^(IV|III|II|I|0)', val)
    if match:
        return match.group(1)
    return 'Unknown'

luad_df['staging_basic'] = luad_df['Stage'].apply(extract_staging_basic)
luad_df['stage_group']   = luad_df['staging_basic'].map({
    'I': 'Early', 'II': 'Early', 'III': 'Late', 'IV': 'Late'
})

print(f'Stage distribution:\n{luad_df["stage_group"].value_counts()}')

In [ ]:
# survival columns — standardize naming and encode status
luad_df = luad_df.rename(columns={'OverallSurvival (Months)': 'months'})
luad_df['status'] = np.where(luad_df['OverallSurvivalStatus'] == '0:LIVING', 0, 1)

# drop samples missing survival data
luad_df = luad_df.dropna(subset=['months', 'status'])

print(f'Samples retained after survival filter: {len(luad_df):,}')

In [ ]:
# binarize mutation columns
# convert "No mutation" / mutation string to 0/1
mut_genes = [
    'ROS1: Mutations (WES)', 'EGFR: Mutations (WES)',
    'ALK: Mutations (WES)',  'BRAF: Mutations (WES)',
    'MET: Mutations (WES)',  'RET: Mutations (WES)',
    'KRAS: Mutations (WES)', 'TP53: Mutations (WES)',
]

for gene in mut_genes:
    if gene in luad_df.columns:
        luad_df[gene] = np.where(
            luad_df[gene].astype(str).str.upper() == 'NO MUTATION', 0, 1
        )

print('Mutation binarization complete')
print(luad_df[mut_genes].sum().sort_values(ascending=False))

In [ ]:
# final analysis dataframe — samples with both expression and survival data
df_rna_meta_mut = luad_df.dropna(subset=['HLA-DRA']).copy()

print(f'Final analysis population: {len(df_rna_meta_mut):,} samples')
print(f'  with survival data: {df_rna_meta_mut["status"].notna().sum():,}')
print(f'  with stage data:    {df_rna_meta_mut["stage_group"].notna().sum():,}')
print(f'  with mutation data: {df_rna_meta_mut["KRAS: Mutations (WES)"].notna().sum():,}')

## Supplemental figure S1a — MHC II gene cluster and overall survival

Patients are clustered into two groups using hierarchical clustering (Ward
linkage) on z-scored expression of the full MHC II gene set. The clustermap
shows the gene expression patterns driving the two clusters. The Kaplan-Meier
curve shows that MHC II High patients have significantly longer overall
survival than MHC II Low patients.

This complements figure 1a by showing the survival difference holds when using
the broader MHC II transcriptional program rather than a single gene (HLA-DRA).
The clustering approach is analogous to the patient classification in
`mhc2-patient-classification.ipynb` but applied to bulk RNA-seq rather than
single-cell data.

**Gene set:** Core MHC II heterodimer genes plus CD74, CIITA, and accessory
molecules (HLA-DMA, HLA-DMB, HLA-DOA, HLA-DOB).

**Method:** Agglomerative hierarchical clustering with Ward linkage on
z-scored expression values. Cluster identity (High vs Low) confirmed by
comparing mean HLA-DRA expression per cluster.

In [ ]:
# MHC II gene set for clustering
mhc2_cluster_genes = [
    'HLA-DRA', 'HLA-DRB1', 'HLA-DRB5', 'HLA-DPA1', 'HLA-DPB1',
    'HLA-DQA1', 'HLA-DQA2', 'HLA-DQB1', 'HLA-DQB2', 'CIITA',
    'HLA-DOA', 'HLA-DOB', 'CD74', 'HLA-DMA', 'HLA-DMB',
]

# subset to genes present in expression matrix
mhc2_cluster_genes = [g for g in mhc2_cluster_genes if g in df_rna_meta_mut.columns]
rna_data = df_rna_meta_mut[mhc2_cluster_genes].dropna()

print(f'Genes used for clustering: {len(mhc2_cluster_genes)}')
print(f'Samples with complete MHC II expression: {len(rna_data):,}')

In [ ]:
# z-score normalize before clustering
# ensures all genes contribute equally regardless of absolute expression level
rna_data_z = pd.DataFrame(
    zscore(rna_data, nan_policy='omit'),
    index=rna_data.index,
    columns=rna_data.columns,
)

In [ ]:
# hierarchical clustering — 2 groups (MHC II High vs Low)
clustering_model = AgglomerativeClustering(n_clusters=2, metric='euclidean', linkage='ward')
clustering_model.fit(rna_data_z)

data_labels      = pd.Series(clustering_model.labels_, index=rna_data_z.index)
survival_cluster = df_rna_meta_mut.loc[rna_data_z.index, ['months', 'status']].copy()
survival_cluster['cluster'] = data_labels

# confirm which cluster is MHC II High by checking mean HLA-DRA per cluster
mean_hladra = rna_data_z.join(data_labels.rename('cluster')).groupby('cluster')['HLA-DRA'].mean()
print(f'Mean HLA-DRA by cluster:\n{mean_hladra}')
print(f'\nCluster {mean_hladra.idxmax()} = MHC II High')

In [ ]:
# exploratory — full MHC II gene set clustering
# shows that using the broader gene set produces the same two clusters
# as the smaller set used in single-cell classification
lut        = {0: cmap_high_low[0], 1: cmap_high_low[1]}
row_colors = data_labels.map(lut)

g = sns.clustermap(
    rna_data_z,
    row_colors=row_colors,
    cmap='vlag', vmin=-2.5, vmax=2.5, center=0,
    method='ward', metric='euclidean',
    xticklabels=True, yticklabels=False,
    figsize=(10, 6),
)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=12)
plt.show()
# note: clustering structure is consistent with the smaller gene set
# (HLA-DRA/DRB1/DQA1/DQB1/DPA1/DPB1 + CD74) used in single-cell classification

In [ ]:
# exploratory — full MHC II gene set clustering
# the broader gene set produces a more balanced High/Low split (~50/50)
# compared to the core gene set (~80/20) because it captures the full
# transcriptional program rather than just the most correlated genes
# both approaches show significant survival differences
print(f'Full gene set cluster sizes:\n{data_labels.value_counts()}')

In [ ]:
# exploratory — KM curve using full MHC II gene set clustering
# confirms survival difference is consistent regardless of gene set used

survival_full = df_rna_meta_mut.loc[rna_data_z.index, ['months', 'status']].copy()
survival_full['cluster'] = data_labels

mean_hladra_full = rna_data_z.join(data_labels.rename('cluster')).groupby('cluster')['HLA-DRA'].mean()
high_cluster_full = mean_hladra_full.idxmax()
low_cluster_full  = 1 - high_cluster_full

T_high_full = survival_full.loc[survival_full['cluster'] == high_cluster_full, 'months']
E_high_full = survival_full.loc[survival_full['cluster'] == high_cluster_full, 'status']
T_low_full  = survival_full.loc[survival_full['cluster'] == low_cluster_full,  'months']
E_low_full  = survival_full.loc[survival_full['cluster'] == low_cluster_full,  'status']

sns.set(font_scale=2.0)
sns.set_style('ticks')

fig, ax = plt.subplots(figsize=(12, 9))

kmf_low_full = KaplanMeierFitter()
kmf_low_full.fit(
    T_low_full, E_low_full,
    label='Cluster 1: MHC class II Low',
    timeline=np.linspace(0, 200),
).plot_survival_function(ax=ax, color=cmap_low_high[0], ci_show=True)

kmf_high_full = KaplanMeierFitter()
kmf_high_full.fit(
    T_high_full, E_high_full,
    label='Cluster 2: MHC class II High',
    timeline=np.linspace(0, 200),
).plot_survival_function(ax=ax, color=cmap_high_low[0], ci_show=True)

add_at_risk_counts(kmf_low_full, kmf_high_full, ax=ax)

results_full = logrank_test(
    T_high_full, T_low_full,
    event_observed_A=E_high_full,
    event_observed_B=E_low_full,
)
p_full = results_full.p_value
ax.add_artist(AnchoredText(f'p = {p_full:.2e}', loc=3, frameon=False))

ax.set_ylim(0, 1)
ax.set_xlabel('Months')
ax.legend(loc='upper right', bbox_to_anchor=(1, 1))
ax.spines[['top', 'right']].set_visible(False)

plt.subplots_adjust(bottom=0.25)
plt.show()  # not saved — exploratory only

print(f'Log-rank p (full gene set) = {p_full:.2e}')

## smaller set of genes 

In [ ]:
# supplemental figure S1a — clustermap using core classification gene set
# matches the gene set used in mhc2-patient-classification.ipynb
classification_genes = [
    'HLA-DRA', 'HLA-DRB1', 'HLA-DQA1', 'HLA-DQB1',
    'HLA-DPA1', 'HLA-DPB1', 'CD74',
]
classification_genes = [g for g in classification_genes if g in df_rna_meta_mut.columns]

rna_data_small   = df_rna_meta_mut[classification_genes].dropna()
rna_data_small_z = pd.DataFrame(
    zscore(rna_data_small, nan_policy='omit'),
    index=rna_data_small.index,
    columns=rna_data_small.columns,
)

# recluster on smaller gene set
clustering_small = AgglomerativeClustering(n_clusters=2, metric='euclidean', linkage='ward')
clustering_small.fit(rna_data_small_z)
labels_small = pd.Series(clustering_small.labels_, index=rna_data_small_z.index)

lut        = {0: cmap_high_low[0], 1: cmap_high_low[1]}
row_colors = labels_small.map(lut)

g = sns.clustermap(
    rna_data_small_z,
    row_colors=row_colors,
    cmap='vlag', vmin=-2.5, vmax=2.5, center=0,
    method='ward', metric='euclidean',
    xticklabels=True, yticklabels=False,
    figsize=(8, 6),
)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=12)
g.fig.savefig(fig_out / 'suppfig1a_mhc2_clustermap.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# note: core gene set produces an unbalanced split — most patients cluster
# as MHC II Low when using only the 7 classification genes
# the full gene set is therefore used for the supplemental KM figure
print(f'Core gene set cluster sizes:\n{labels_small.value_counts()}')

In [ ]:
# supplemental figure S1a — KM curve using core classification gene set
survival_small = df_rna_meta_mut.loc[rna_data_small_z.index, ['months', 'status']].copy()
survival_small['cluster'] = labels_small

mean_hladra_small = rna_data_small_z.join(labels_small.rename('cluster')).groupby('cluster')['HLA-DRA'].mean()
high_cluster_small = mean_hladra_small.idxmax()
low_cluster_small  = 1 - high_cluster_small

T_high_small = survival_small.loc[survival_small['cluster'] == high_cluster_small, 'months']
E_high_small = survival_small.loc[survival_small['cluster'] == high_cluster_small, 'status']
T_low_small  = survival_small.loc[survival_small['cluster'] == low_cluster_small,  'months']
E_low_small  = survival_small.loc[survival_small['cluster'] == low_cluster_small,  'status']

sns.set(font_scale=2.0)
sns.set_style('ticks')

fig, ax = plt.subplots(figsize=(12, 9))

kmf_low_small = KaplanMeierFitter()
kmf_low_small.fit(
    T_low_small, E_low_small,
    label='Cluster 1: MHC class II Low',
    timeline=np.linspace(0, 200),
).plot_survival_function(ax=ax, color=cmap_low_high[0], ci_show=True)

kmf_high_small = KaplanMeierFitter()
kmf_high_small.fit(
    T_high_small, E_high_small,
    label='Cluster 2: MHC class II High',
    timeline=np.linspace(0, 200),
).plot_survival_function(ax=ax, color=cmap_high_low[0], ci_show=True)

add_at_risk_counts(kmf_low_small, kmf_high_small, ax=ax)

results_small = logrank_test(
    T_high_small, T_low_small,
    event_observed_A=E_high_small,
    event_observed_B=E_low_small,
)
p_small = results_small.p_value
ax.add_artist(AnchoredText(f'p = {p_small:.2e}', loc=3, frameon=False))

ax.set_ylim(0, 1)
ax.set_xlabel('Months')
ax.legend(loc='upper right', bbox_to_anchor=(1, 1))
ax.spines[['top', 'right']].set_visible(False)

plt.subplots_adjust(bottom=0.25)
plt.savefig(fig_out / 'suppfig1a_mhc2_cluster_km.pdf', bbox_inches='tight', dpi=300)
plt.show()

print(f'Log-rank p (core gene set) = {p_small:.2e}')

### Supplemental figure S1a — justification of gene set choice

Two clustering approaches were tested to define MHC II High and Low patient
groups:

1. **Full MHC II gene set** (15 genes including HLA-DQA1/2, HLA-DQB1/2,
   HLA-DRB5, HLA-DOA/B, HLA-DMA/B, CIITA) — produces a balanced ~50/50
   split, log-rank p = 1.37×10⁻⁹
2. **Core classification gene set** (HLA-DRA, HLA-DRB1, HLA-DQA1, HLA-DQB1,
   HLA-DPA1, HLA-DPB1, CD74) — produces an unbalanced ~80/20 split,
   log-rank p = 8.76×10⁻⁹

Both splits show highly significant survival differences, confirming the
result is robust to gene set choice.

The clustermap reveals that HLA-DQA1/DQB1 and HLA-DRB1 follow a partially
distinct expression program from HLA-DRA, HLA-DPA1, HLA-DPB1, and CD74.
This may reflect differential regulation of the DR, DP, and DQ loci. For
the main figure (figure 1a), HLA-DRA expression alone is used as it is the
most strongly expressed MHC II gene, is broadly representative of the MHC II
High transcriptional state, and simplifies interpretation for a clinical
audience.

## Figure 1a — HLA-DRA expression and overall survival

Patients are split into HLA-DRA High and Low groups by median expression.
HLA-DRA is used as the primary MHC II biomarker as it is the most strongly
expressed MHC II gene and broadly representative of the MHC II High
transcriptional state. Log-rank test p-value shown on figure.

**Method:** Median split on HLA-DRA z-score expression. Kaplan-Meier
estimator with 95% confidence intervals. Log-rank test for survival
difference between groups.

In [ ]:
# split patients into HLA-DRA High vs Low by median expression
dra_df = df_rna_meta_mut[['HLA-DRA', 'months', 'status']].dropna().copy()
dra_df['HLA_DRA_bin'] = np.where(dra_df['HLA-DRA'] >= dra_df['HLA-DRA'].median(), 1, 0)

T_high = dra_df.loc[dra_df['HLA_DRA_bin'] == 1, 'months']
E_high = dra_df.loc[dra_df['HLA_DRA_bin'] == 1, 'status']
T_low  = dra_df.loc[dra_df['HLA_DRA_bin'] == 0, 'months']
E_low  = dra_df.loc[dra_df['HLA_DRA_bin'] == 0, 'status']

print(f'HLA-DRA High: {len(T_high):,} samples')
print(f'HLA-DRA Low:  {len(T_low):,} samples')

In [ ]:
sns.set(font_scale=2.0)
sns.set_style('ticks')

fig, ax = plt.subplots(figsize=(12, 9))

kmf_low = KaplanMeierFitter()
kmf_low.fit(
    T_low, E_low,
    label='Cluster 1: HLA-DRA Low',
    timeline=np.linspace(0, 200),
).plot_survival_function(ax=ax, color=cmap_low_high[0], ci_show=True)

kmf_high = KaplanMeierFitter()
kmf_high.fit(
    T_high, E_high,
    label='Cluster 2: HLA-DRA High',
    timeline=np.linspace(0, 200),
).plot_survival_function(ax=ax, color=cmap_high_low[0], ci_show=True)

add_at_risk_counts(kmf_low, kmf_high, ax=ax)

results = logrank_test(T_high, T_low, event_observed_A=E_high, event_observed_B=E_low)
p       = results.p_value
ax.add_artist(AnchoredText(f'p = {p:.2e}', loc=3, frameon=False))

ax.set_ylim(0, 1)
ax.set_xlabel('Months')
ax.legend(loc='upper right', bbox_to_anchor=(1, 1))
ax.spines[['top', 'right']].set_visible(False)

plt.subplots_adjust(bottom=0.25)
plt.savefig(fig_out / 'figure1a_hladra_km.pdf', bbox_inches='tight', dpi=300)
plt.show()

print(f'Log-rank p = {p:.2e}')

## Figure 1b — Multivariate Cox proportional hazards forest plot

HLA-DRA expression remains a significant predictor of overall survival after
adjustment for clinical covariates (age, sex, stage, TMB, MSI) and immune
gene expression (CD274, CD8A, CD3D). Each variable is binarized at the median
(continuous variables) or coded as mutated vs wild-type (mutation variables).
Hazard ratios are oriented so HR > 1 always represents worse survival.
Color coding distinguishes clinical features, gene expression, and mutations.

**Method:** Univariate Cox proportional hazards models fit separately for each
variable. Ridge penalization (λ=0.1) applied for numerical stability. HR < 1
are flipped so all comparisons read as worse vs better outcome.

In [ ]:
# binary codings — 1 = hypothesized worse outcome
df = df_rna_meta_mut.copy()
df['Sex_bin']      = np.where(df['Sex'] == 'Male', 1, 0)
df['Age_bin']      = np.where(df['AgeAtDiagnosis'] >= df['AgeAtDiagnosis'].median(), 1, 0)
df['Stage_bin']    = np.where(df['stage_group'] == 'Late', 1, 0)
df['TMB_bin']      = np.where(df['TumorMutationalBurden'] >= df['TumorMutationalBurden'].median(), 1, 0)
df['CD274_bin']    = np.where(df['CD274'] >= df['CD274'].median(), 1, 0)
df['HLA_DRA_bin']  = np.where(df['HLA-DRA'] >= df['HLA-DRA'].median(), 1, 0)
df['CD8A_bin']     = np.where(df['CD8A'] >= df['CD8A'].median(), 1, 0)
df['CD3D_bin']     = np.where(df['CD3D'] >= df['CD3D'].median(), 1, 0)
df['MSI_bin']      = np.where(df['MicroSatelliteInstability'] >= df['MicroSatelliteInstability'].median(), 1, 0)

mut_genes = [
    'TP53: Mutations (WES)', 'KRAS: Mutations (WES)', 'EGFR: Mutations (WES)',
    'ALK: Mutations (WES)',  'BRAF: Mutations (WES)', 'MET: Mutations (WES)',
    'ROS1: Mutations (WES)', 'RET: Mutations (WES)',
]
for gene in mut_genes:
    clean = gene.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '')
    df[f'{clean}_bin'] = np.where(
        df[gene].astype(str).str.upper().isin(['MUTATED', '1', 'TRUE']), 1, 0
    )

# metadata — display labels for each binary variable
var_meta = {
    'Sex_bin':     ('Male', 'Female', 'Sex'),
    'Age_bin':     ('Older (≥ median)', 'Younger (< median)', 'Age'),
    'Stage_bin':   ('Late', 'Early', 'Stage'),
    'TMB_bin':     ('High (≥ median)', 'Low (< median)', 'Tumor Mutational Burden'),
    'CD274_bin':   ('High (≥ median)', 'Low (< median)', 'CD274'),
    'HLA_DRA_bin': ('High (≥ median)', 'Low (< median)', 'HLA-DRA'),
    'CD8A_bin':    ('High (≥ median)', 'Low (< median)', 'CD8A'),
    'CD3D_bin':    ('High (≥ median)', 'Low (< median)', 'CD3D'),
    'MSI_bin':     ('High (≥ median)', 'Low (< median)', 'MicroSatellite Instability'),
}
for gene in mut_genes:
    clean = gene.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '')
    gene_short = gene.split(':')[0]
    var_meta[f'{clean}_bin'] = ('Mutated', 'WT', gene_short)

order = list(var_meta.keys())
print(f'Variables defined: {len(order)}')
print([c for c in df.columns if c.endswith('_bin')])

In [ ]:
# multivariate Cox model — all covariates fit simultaneously
# HRs are adjusted for all other variables in the model
covariates = [
    'HLA_DRA_bin', 'Sex_bin', 'Age_bin', 'Stage_bin', 'TMB_bin',
    'CD274_bin', 'CD8A_bin', 'CD3D_bin', 'MSI_bin',
] + [
    f"{g.replace(' ', '_').replace(':', '').replace('(', '').replace(')', '')}_bin"
    for g in mut_genes
]

sub = df[covariates + ['months', 'status']].dropna()
print(f'Samples in multivariate model: {len(sub):,}')

cph = CoxPHFitter(penalizer=0.1)
cph.fit(sub, duration_col='months', event_col='status')
cph.print_summary()

In [ ]:
# extract results and build display labels
summary = cph.summary.reset_index().rename(columns={
    'index':                  'covariate',
    'exp(coef)':              'HR',
    'exp(coef) lower 95%':   'CI_lower',
    'exp(coef) upper 95%':   'CI_upper',
    'p':                      'pval',
})

results = []
for _, row in summary.iterrows():
    col       = row['covariate']
    HR, lo, hi, p = row['HR'], row['CI_lower'], row['CI_upper'], row['pval']

    if col not in var_meta:
        continue

    pos, neg, pretty = var_meta[col]

    # flip so HR > 1 = worse outcome
    if HR < 1:
        HR, lo, hi = 1/HR, 1/hi, 1/lo
        label = f'{pretty} ({neg} vs {pos})'
    else:
        label = f'{pretty} ({pos} vs {neg})'

    results.append((label, HR, lo, hi, p))

results_df = pd.DataFrame(results, columns=['Variable', 'HR', 'CI_lower', 'CI_upper', 'pval'])
results_df = results_df.sort_values('HR', ascending=False).reset_index(drop=True)

# classify and color
results_df['Group'] = results_df['Variable'].apply(classify_variable)
results_df['Color'] = results_df['Group'].map(group_colors)

print(results_df[['Variable', 'HR', 'pval']].to_string())

In [ ]:
# forest plot — adjusted HRs from multivariate model
sns.set(font_scale=1.3)
sns.set_style('ticks')

fig, ax = plt.subplots(figsize=(8, 6))

y      = np.arange(len(results_df))
x      = results_df['HR'].values
xerr   = np.vstack([x - results_df['CI_lower'].values, results_df['CI_upper'].values - x])
colors = results_df['Color'].values

for i in range(len(results_df)):
    ax.errorbar(
        x[i], y[i],
        xerr=[[xerr[0, i]], [xerr[1, i]]],
        fmt='o', color=colors[i], ecolor='gray', elinewidth=1.2,
        capsize=3, markersize=8,
    )

ax.axvline(1, color='black', linestyle='--', linewidth=1)
ax.set_xscale('log')
ax.set_xlim(0.6, results_df['CI_upper'].max() * 1.3)
ax.set_yticks(y)
ax.set_yticklabels(results_df['Variable'], fontsize=12)
ax.set_xlabel('Hazard Ratio (adjusted)', fontsize=12)
ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)

handles = [
    plt.Line2D([0], [0], color=c, marker='o', linestyle='', label=g)
    for g, c in group_colors.items()
]
ax.legend(handles=handles, title='Feature Group', frameon=True,
          loc='lower right', fontsize=12)

plt.tight_layout()
plt.savefig(fig_out / 'figure1b_cox_forest.pdf', bbox_inches='tight', dpi=300)
plt.show()

# save results table
results_df['HR (95% CI)'] = results_df.apply(
    lambda r: f'{r.HR:.2f} [{r.CI_lower:.2f}-{r.CI_upper:.2f}]', axis=1
)
results_df[['Variable', 'HR (95% CI)', 'pval']].to_csv(
    table_out / 'figure1b_cox_results.csv', index=False
)
print(f'Saved → {table_out / "figure1b_cox_results.csv"}')